In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "easyocr", "spacy", "pandas", "openpyxl", "pillow", "tqdm"])
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

0

In [ ]:
import os, re, json
import easyocr
import spacy
import pandas as pd
from PIL import Image
from tqdm import tqdm

#  Paths 
DATA_DIR   = "data/raw"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

#  Load models 
print("Loading EasyOCR...")
reader = easyocr.Reader(["en"], gpu=False)  
print("Loading spaCy...")
nlp = spacy.load("en_core_web_sm")
print("Models ready!\n")


# PREPROCESSING
def preprocess_image(img_path):
    
    img = Image.open(img_path).convert("RGB")
    
    #Pillow
    w, h = img.size
    if w < 800:
        img = img.resize((800, int(h * 800 / w)), Image.LANCZOS)
    return img


# OCR
def run_ocr(img_path):
    
    img = preprocess_image(img_path)
    
    temp_path = "temp_preprocessed.png"
    img.save(temp_path)
    results = reader.readtext(temp_path, detail=0, paragraph=True)
    os.remove(temp_path)
    return results


# FIELD EXTRACTION

#  Email 
def extract_email(text):
    match = re.findall(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", text)
    return match[0] if match else ""

# Phone 
def extract_phone(text):
    match = re.findall(r"[\+\(]?[1-9][0-9\s\-\(\)]{7,}[0-9]", text)
    return match[0].strip() if match else ""

#  Name 
def extract_name(lines, doc):
   
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            return ent.text
 
    for line in lines:
        if len(line.strip()) > 2:
            return line.strip()
    return ""

#  Location
def extract_location(doc):
    for ent in doc.ents:
        if ent.label_ in ["GPE", "LOC"]:
            return ent.text
    return ""

# Section detector
SECTION_KEYWORDS = {
    "skills":     ["skill", "skills", "technical skills", "core competencies",
                   "technologies", "tools", "expertise"],
    "education":  ["education", "academic", "qualification", "degree",
                   "university", "college", "school"],
    "experience": ["experience", "work experience", "employment", "career",
                   "professional experience", "internship", "projects"],
}

def detect_sections(lines):
    
    sections = {k: [] for k in SECTION_KEYWORDS}
    current_section = None

    for line in lines:
        line_lower = line.lower().strip()
        matched = False
        for section, keywords in SECTION_KEYWORDS.items():
            if any(kw in line_lower for kw in keywords):
                current_section = section
                matched = True
                break
        if not matched and current_section:
            sections[current_section].append(line.strip())

    return sections

def clean_section(lines):
   
    seen = set()
    clean = []
    for l in lines:
        l = l.strip()
        if l and l not in seen:
            seen.add(l)
            clean.append(l)
    return " | ".join(clean)


#  MAIN PIPELINE

SUPPORTED = (".jpg", ".jpeg", ".png", ".webp")
image_files = [f for f in os.listdir(DATA_DIR)
               if f.lower().endswith(SUPPORTED)]

print(f"Found {len(image_files)} resume images\n")

records = []

for fname in tqdm(image_files):
    img_path = os.path.join(DATA_DIR, fname)

    try:
        # OCR
        lines = run_ocr(img_path)
        full_text = " ".join(lines)

        # spaCy
        doc = nlp(full_text[:10000])  

        # Extract fields
        name     = extract_name(lines, doc)
        email    = extract_email(full_text)
        phone    = extract_phone(full_text)
        location = extract_location(doc)

        # Section detection
        sections   = detect_sections(lines)
        skills     = clean_section(sections["skills"])
        education  = clean_section(sections["education"])
        experience = clean_section(sections["experience"])

        records.append({
            "file":       fname,
            "name":       name,
            "email":      email,
            "phone":      phone,
            "location":   location,
            "skills":     skills,
            "education":  education,
            "experience": experience,
        })

    except Exception as e:
        print(f"Error on {fname}: {e}")
        records.append({"file": fname, "name": "", "email": "",
                        "phone": "", "location": "", "skills": "",
                        "education": "", "experience": ""})

# EXPORT TO EXCEL
df = pd.DataFrame(records)
output_path = os.path.join(OUTPUT_DIR, "resume_output.xlsx")
df.to_excel(output_path, index=False)
print(f"\nDone! Saved to {output_path}")
print(f"Total resumes processed: {len(df)}")

# QUICK METRICS


print("\n" + "="*50)
print("  EXTRACTION METRICS")
print("="*50)
fields = ["name", "email", "phone", "location", "skills", "education", "experience"]
for f in fields:
    filled = df[f].apply(lambda x: 1 if str(x).strip() else 0).sum()
    pct    = filled / len(df) * 100
    print(f"  {f:<12} extracted in {filled}/{len(df)} resumes → {pct:.1f}%")

print("\nOpen output/resume_output.xlsx to see your structured data!")

Using CPU. Note: This module is much faster with a GPU.


Loading EasyOCR...
Loading spaCy...
Models ready!

Found 50 resume images



  0%|          | 0/50 [00:00<?, ?it/s]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  2%|▏         | 1/50 [00:29<24:16, 29.73s/it]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  4%|▍         | 2/50 [00:45<17:25, 21.78s/it]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  6%|▌         | 3/50 [01:06<16:30, 21.08s/it]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerato


Done! Saved to output\resume_output.xlsx
Total resumes processed: 50

  EXTRACTION METRICS
  name         extracted in 50/50 resumes → 100.0%
  email        extracted in 19/50 resumes → 38.0%
  phone        extracted in 46/50 resumes → 92.0%
  location     extracted in 41/50 resumes → 82.0%
  skills       extracted in 40/50 resumes → 80.0%
  education    extracted in 36/50 resumes → 72.0%
  experience   extracted in 38/50 resumes → 76.0%

Open output/resume_output.xlsx to see your structured data!


In [ ]:
import os, re, json
import pandas as pd



df = pd.read_excel("output/resume_output.xlsx")

# Email
def fix_email(email):
    if pd.isna(email) or not email:
        return ""
    email = str(email)
   
    email = re.sub(r'_com$', '.com', email)
    email = re.sub(r'_net$', '.net', email)
    email = re.sub(r'_org$', '.org', email)
    email = re.sub(r'gmail_', 'gmail.', email)
    email = re.sub(r'yahoo_', 'yahoo.', email)
    email = re.sub(r'outlook_', 'outlook.', email)
   
    if re.match(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z]{2,}", email):
        return email.lower().strip()
    return ""


def fix_phone(phone):
    if pd.isna(phone) or not phone:
        return ""
    phone = str(phone)
    
    digits = re.sub(r'\D', '', phone)
    if len(digits) < 7 or len(digits) > 15:
        return ""
   
    if re.match(r'^(19|20)\d{2}$', digits):
        return ""
    return phone.strip()


TITLE_WORDS = ["senior", "junior", "student", "manager", "engineer",
               "developer", "designer", "analyst", "director", "officer",
               "intern", "assistant", "lead", "head", "chief"]

def fix_name(name):
    if pd.isna(name) or not name:
        return ""
    name = str(name).strip()
    words = name.split()
    # take max 3 words
    words = words[:3]
    # remove title words
    words = [w for w in words if w.lower() not in TITLE_WORDS]
    return " ".join(words).strip()


NOISE_PATTERNS = [
    r'\b(address|phone|email|contact|reference|date of birth|dob|'
    r'nationality|linkedin|github|website|www|http)\b',
    r'\d{1,2}[\/\-]\d{1,2}[\/\-]\d{2,4}',  # dates
    r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z]{2,}',  # emails
    r'\+?[\d\s\-\(\)]{7,}',  
]

def fix_skills(skills):
    if pd.isna(skills) or not skills:
        return ""
    parts = str(skills).split(" | ")
    clean = []
    for part in parts:
        part = part.strip()
        
        if len(part) > 60:
            continue
        
        noisy = False
        for pat in NOISE_PATTERNS:
            if re.search(pat, part, re.IGNORECASE):
                noisy = True
                break
        if not noisy and len(part) > 1:
            clean.append(part)
    return " | ".join(clean[:15])  


LANG_WORDS = ["english", "german", "french", "arabic", "spanish",
              "chinese", "hindi", "language", "languages"]

def fix_education(edu):
    if pd.isna(edu) or not edu:
        return ""
    parts = str(edu).split(" | ")
    clean = []
    for part in parts:
        part = part.strip()
       
        words = part.lower().split()
        lang_count = sum(1 for w in words if w in LANG_WORDS)
        if lang_count / max(len(words), 1) > 0.4:
            continue
        if len(part) > 2:
            clean.append(part)
    return " | ".join(clean)


df_clean = df.copy()
df_clean["name"]      = df["name"].apply(fix_name)
df_clean["email"]     = df["email"].apply(fix_email)
df_clean["phone"]     = df["phone"].apply(fix_phone)
df_clean["skills"]    = df["skills"].apply(fix_skills)
df_clean["education"] = df["education"].apply(fix_education)


df_clean.to_excel("output/resume_output_cleaned.xlsx", index=False)
print("Saved cleaned file → output/resume_output_cleaned.xlsx")


FIELDS = ["name", "email", "phone", "location", "skills", "education", "experience"]
print("\n" + "="*55)
print("  RAW vs CLEANED — Field Fill Rate")
print("="*55)
print(f"  {'Field':<12} {'Raw':>8}  {'Cleaned':>8}")
print("  " + "-"*35)
for f in FIELDS:
    raw     = df[f].apply(lambda x: 1 if str(x).strip() and str(x) != 'nan' else 0).mean() * 100
    cleaned = df_clean[f].apply(lambda x: 1 if str(x).strip() and str(x) != 'nan' else 0).mean() * 100
    print(f"  {f:<12} {raw:>7.1f}%  {cleaned:>7.1f}%")

Saved cleaned file → output/resume_output_cleaned.xlsx

  RAW vs CLEANED — Field Fill Rate
  Field             Raw   Cleaned
  -----------------------------------
  name           100.0%    100.0%
  email           38.0%     38.0%
  phone           92.0%     86.0%
  location        82.0%     82.0%
  skills          80.0%     64.0%
  education       72.0%     70.0%
  experience      76.0%     76.0%


In [ ]:
import pandas as pd
import re
import numpy as np

df = pd.read_excel("output/resume_output_cleaned.xlsx")


EDU_KEYWORDS = ["bachelor", "master", "phd", "b.sc", "m.sc", "b.tech", "m.tech",
                "degree", "diploma", "university", "college", "school", "gpa",
                "graduation", "undergraduate", "postgraduate", "b.e", "m.e",
                "higher secondary", "sslc", "hsc", "ssc", "gce", "a level",
                "institute", "faculty", "department", "major", "minor",
                "science", "engineering", "commerce", "arts", "management"]

EXP_KEYWORDS = ["company", "inc", "ltd", "pvt", "corp", "intern", "developer",
                "designer", "analyst", "manager", "engineer", "worked", "present",
                "responsible", "developed", "managed", "led", "built", "created"]

NOISE_WORDS  = ["address", "phone", "email", "@", "www", "http",
                "contact", "linkedin", "github", "reference"]


def clean_education_final(edu_text):
    if pd.isna(edu_text) or not edu_text or str(edu_text) == 'nan':
        return ""
    parts = str(edu_text).split(" | ")
    clean = []
    for part in parts:
        part = part.strip()
        if len(part) < 3:
            continue
        if any(noise in part.lower() for noise in NOISE_WORDS):
            continue
        has_edu_kw = any(kw in part.lower() for kw in EDU_KEYWORDS)
        has_year   = bool(re.search(r'\b(19|20)\d{2}\b', part))
        if has_edu_kw or (has_year and len(part) < 150):
            clean.append(part)
    if not clean and parts:
        clean = [p.strip() for p in parts[:2] if len(p.strip()) > 3]
    return " | ".join(clean[:5])


def clean_experience_final(exp_text):
    if pd.isna(exp_text) or not exp_text or str(exp_text) == 'nan':
        return ""
    parts = str(exp_text).split(" | ")
    clean = []
    for part in parts:
        part = part.strip()
        if len(part) < 3:
            continue
        if any(noise in part.lower() for noise in NOISE_WORDS):
            continue
        has_edu_only = any(kw in part.lower() for kw in
                          ["bachelor", "master", "phd", "b.tech", "m.tech", "b.sc"])
        has_company  = any(kw in part.lower() for kw in EXP_KEYWORDS)
        if has_edu_only and not has_company:
            continue
        clean.append(part)
    return " | ".join(clean[:5])


df["education"]  = df["education"].apply(clean_education_final)
df["experience"] = df["experience"].apply(clean_experience_final)

df.to_excel("output/resume_output_final_v2.xlsx", index=False)
print("Done! Saved → output/resume_output_final_v2.xlsx")

# ── Quick check ───────────────────────────────────────────────
for field in ["education", "experience"]:
    filled = df[field].apply(lambda x: 1 if str(x).strip() and str(x) != 'nan' else 0).sum()
    print(f"{field}: {filled}/50 filled")

Done! Saved → output/resume_output_final_v2.xlsx
education: 35/50 filled
experience: 38/50 filled


In [6]:
import numpy as np

def validate_email(val):
    if not val or str(val) == 'nan': return 0
    return 1 if re.match(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z]{2,}", str(val)) else 0

def validate_phone(val):
    if not val or str(val) == 'nan': return 0
    digits = re.sub(r'\D', '', str(val))
    return 1 if 7 <= len(digits) <= 15 else 0

def validate_name(val):
    if not val or str(val) == 'nan': return 0
    words = str(val).strip().split()
    return 1 if 1 <= len(words) <= 5 and len(str(val)) < 50 else 0

def validate_location(val):
    if not val or str(val) == 'nan': return 0
    return 1 if 2 <= len(str(val)) <= 100 else 0

def validate_section(val):
    if not val or str(val) == 'nan': return 0
    return 1 if len(str(val).strip()) > 10 else 0

validators = {
    "name":       validate_name,
    "email":      validate_email,
    "phone":      validate_phone,
    "location":   validate_location,
    "skills":     validate_section,
    "education":  validate_section,
    "experience": validate_section,
}

print("\n" + "="*60)
print("  FINAL METRICS — Format-Validated Accuracy")
print("="*60)
print(f"  {'Field':<12} {'Extracted':>10}  {'Valid':>8}  {'Validity%':>10}")
print("  " + "-"*48)

summary = {}
for field, validator in validators.items():
    extracted = df[field].apply(lambda x: 1 if str(x).strip() and str(x) != 'nan' else 0).sum()
    valid     = df[field].apply(validator).sum()
    ext_pct   = extracted / len(df) * 100
    val_pct   = valid / len(df) * 100
    summary[field] = {"extracted": int(extracted), "valid": int(valid),
                      "extraction_rate": round(ext_pct, 1),
                      "validity_rate":   round(val_pct, 1)}
    print(f"  {field:<12} {extracted:>5}/{len(df):<5} {ext_pct:>5.1f}%"
          f"  {valid:>4}/{len(df):<4} {val_pct:>7.1f}%")

avg_ext = np.mean([v["extraction_rate"] for v in summary.values()])
avg_val = np.mean([v["validity_rate"]   for v in summary.values()])
print("  " + "-"*48)
print(f"  {'Average':<12} {'':>10}  {avg_ext:>7.1f}%  {'':>4}     {avg_val:>7.1f}%")

print(f"""
  Extraction Rate = field was found in the resume (coverage)
  Validity Rate   = extracted value passes format validation
                    (proxy for precision / accuracy)

RESUME BULLET:
──────────────────────────────────────────────────────
Built a resume image extraction pipeline on 50 resume
images using EasyOCR + spaCy NER; achieved {summary['phone']['validity_rate']}% phone,
{summary['name']['validity_rate']}% name and {summary['education']['validity_rate']}% education validity via
format-based validation; avg field extraction rate {avg_ext:.0f}%.
──────────────────────────────────────────────────────
""")


  FINAL METRICS — Format-Validated Accuracy
  Field         Extracted     Valid   Validity%
  ------------------------------------------------
  name            50/50    100.0%    50/50     100.0%
  email           19/50     38.0%    19/50      38.0%
  phone           43/50     86.0%    43/50      86.0%
  location        41/50     82.0%    41/50      82.0%
  skills          32/50     64.0%    31/50      62.0%
  education       35/50     70.0%    30/50      60.0%
  experience      38/50     76.0%    38/50      76.0%
  ------------------------------------------------
  Average                     73.7%              72.0%

  Extraction Rate = field was found in the resume (coverage)
  Validity Rate   = extracted value passes format validation
                    (proxy for precision / accuracy)

RESUME BULLET:
──────────────────────────────────────────────────────
Built a resume image extraction pipeline on 50 resume
images using EasyOCR + spaCy NER; achieved 86.0% phone,
100.0% name and